### 자동 미분
- 신경망을 학습할 때 가장 자주 사용되는 알고리즘 -> 역전파
- 가중치는 주어진 parameter에 대한 손실 함수의 변화도(gradient)에 따라 조정됨

- 이러한 변화도를 계산하기 위해 PyTorch에서 `torch.autograd`라고 불리는 자동 미분 엔진이 내장되어있음
    - 모든 계산 그래프에 대한 변화도의 자동 계산을 지원

- 입력 `x`, 매개변수`w`와 `b`, 그리고 일부 손실 함수가 있는 가장 간단한 단일 계층 신경망을 가정 

In [ ]:
import torch

x = torch.rand(5) # 입력 텐서
y = torch.zeros(3) # 타겟 텐서
w = torch.randn(5, 3, requires_grad=True) # 가중치 텐서
b = torch.randn(3, requires_grad=True) # 편향 텐서
z = torch.matmul(x, w) + b # 선형 변환
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y) # 손실 계산

__Tensor, Function과 연산 그래프__
- 신경망에서 w, b는 최적화를 해야 하는 매개변수이다
- 따라서 이러한 변수들에 대한 손실 함수의 변화도를 계산할 수 있어야 함
    - 이를 위해 텐서에서 `requires_grad` 속성을 설정함

> 참고 
> `requires_grad`의 값은 텐서를 생성할 때 설정하거나, 나중에 `x.requires_grad_(True)` 메소드를 사용하여 나중에 설정할 수 있음

- 연산 그래프를 구성하기 위해 텐서에 적용하는 함수는 사실 `Function` 클래스의 객체임
- 이 객체는 순전파 방향으로 함수를 계산하는 방법, 역방향 전파 단계에서 도함수를 계산하는 방법을 알고있음
- 역방향 전파 함수에 대한 참조는 텐서의 `grad_fn`속성에 저장됨

In [ ]:
print(f"Gradient function for z = {z.grad_fn}") # 선형 변환의 그래디언트 함수 출력
print(f"Gradient function for loss = {loss.grad_fn}") # 손실의 그래디언트 함수 출력

Gradient function for z = <AddBackward0 object at 0x11fe40970>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x103733ca0>


__Gradient 계산하기__
- 신경망에서 매개변수의 가중치를 최적화하기 위해 parameter에 대한 손실함수의 도함수를 계산해야 함.
$$ \frac{\delta loss}{\delta w} 와 \frac{\delta loss}{\delta b} $$
- x, y의 일부 고정값에서 위의 식이 필요함
- 이러한 도하수를 계산하기 위해, `loss.backward()`를 호출한 다음 `w.grad`와, `b.grad` 에서 값을 가져옴

In [ ]:
loss.backward() # 역전파 수행
print(w.grad)
print(b.grad)

tensor([[0.0255, 0.0173, 0.0259],
        [0.0164, 0.0111, 0.0166],
        [0.0352, 0.0238, 0.0357],
        [0.0332, 0.0225, 0.0337],
        [0.0047, 0.0032, 0.0048]])
tensor([0.1274, 0.0863, 0.1295])


__변화도 추적 멈추기__
- 기본적으로 `requires_grad=True` 인 모든 텐서들은 연산 기록을 추적하고 변화도 계산을 지원함
- 그러나 모델을 학습한 뒤 입력 데이터를 단순히 적용하기만 하는 경우와 같이 순전파 연산만 필요한 경우에는, 이러한 기능이 필요 없음
- 연산 코드를 `torch.no_grad()` 블록으로 둘러싸 연산 추적을 멈출 수 있음

In [8]:
z = torch.matmul(x, w) + b # 선형 변환
print(z.requires_grad) 

with torch.no_grad(): # 연산 추적 멈추기
    z = torch.matmul(x, w) + b # 선형 변환
print(z.requires_grad)

True
False


- 동일한 결과를 얻는 다른 방법은 텐서에 `detach()`메소드를 사용하는 것

- 변화도 추적을 멈춰야 하는 이유
    - 신경망의 일부 parameter를 고정된 매개변수(frozen parameter)로 표시
    - 변화도를 추적하지 않는 텐서의 연산이 더 효율적 -> 순전파 단계만 수행할 때 연산 속도가 향상됨

In [ ]:
z = torch.matmul(x, w) + b # 선형 변환
z_det = z.detach() # 변화도 추적 멈추기
print(z_det.requires_grad)

False


__연산 그래프에 대한 추가 정보__
- 개념적으로 autograd는 텐서 및 실행된 모든 연산들의 기록을 `Function` 객체로 구성된 방향성 비순환 그래프 (DAG: Directed Acyclic Graph)에 저장 함
- 이 방향성 비순환 그래프의 잎(leave)은 입력 텐서, 뿌리(root)는 결과 텐서
- 이 그래프를 뿌리에서부터 잎까지 추적하면 연쇄 법칙에 따라 변화도를 자동으로 계산할 수 있음

- 순전파 단계에서, autograd는 다음 두 가지 작업을 동시에 수행
    - 요청된 연산을 수행하여 결과 텐서를 계산
    - DAG에 연산의 변화도 기능(gradient function)을 유지(maintain) 함

- 역전파 단계는 DAG 뿌리에서 `.backward()`가 호출될 때 시작됨
    - 각 `.grad_fn`으로부터 변화도 계산
    - 각 텐서의 `.grad` 속성에 계산 결과를 쌓고
    - 연쇄 법칙을 사용 -> 모든 잎 텐서들까지 전파함

>참고
:PyTorch 에서 DAG들은 동적임.
주목해야 할 점은 그래프가 처음부터 다시 생성된다는 것
매번 `.backward()`가 호출되고 나면, autograd는 새로운 그래프를 채우기(populate) 시작함
이러한 점 때문에 모델에서 흐름 제어 구문들을 사용할 수 있음
매번 반복할 때 필요하면 shape, size, operation을 바꿀 수 있음


__선택적으로 읽기(Optional Reading)__
- 대부분의 경우, 스칼라 손실 함수를 가지고 일반 매개변수와 관련한 변화도를 계산해야 함
- 그러나 출력 함수가 임의의 텐서인 경우가 있음 -> 이때 PyTorch는 실제 변화도가 아닌, 야코비안 곱(Jacobian product)을 계산함
- 야코비안 행렬 자체를 계산하는 대신, PyTorch는 주어진 입력 벡터에 대한 야코비안 곱을 계산해야 함
    - 이 과정은 v를 인자로 `backward`를 호출하면 이루어짐 -> v의 크기는 곱을 계산하려고 하는 원래 텐서의 크기와 같아야 함

In [9]:
inp = torch.eye(4, 5, requires_grad=True) # 4x5 단위 행렬
out = (inp+1).pow(2).t() # 몇 가지 연산 수행
out.backward(torch.ones_like(out), retain_graph=True) # 역전파 수행
print(f"First call\n{inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True) # 역전파 수행
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_() # 그래디언트 0으로 초기화
out.backward(torch.ones_like(out), retain_graph=True) # 역전파 수행
print(f"\nCall after zeroing gradients\n{inp.grad}")

First call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])

Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])
